# Descriptive-Title
Felix Zaussinger | XX.YY.ZZZZ

## Core Analysis Goal(s)
1.
2.
3.

## Key Insight(s)
1.
2.
3.

In [69]:
import os
import sys
import logging
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

from src import utils
#import mapping_career_causeways

# OPTIONAL: Load the "autoreload" extension so that code can change
%load_ext autoreload

# OPTIONAL: always reload modules so that as you change code in src, it gets loaded
%autoreload 2

# Settings
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

sns.set_context("poster")
sns.set(rc={'figure.figsize': (16, 9.)})
sns.set_style("ticks")

pd.set_option("display.max_rows", 120)
pd.set_option("display.max_columns", 120)

logging.basicConfig(level=logging.INFO, stream=sys.stdout)

# load paths
useful_paths = utils.UsefulPaths(fn_config_path="paths_config.yml")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


Configuration files

In [70]:
from src.data.framework import Crosswalks, Onet
crosswalks = Crosswalks()
onet = Onet()

#### Crosswalk IT (CP 2011) - ESCO

Mapping project name,AI4ESCO
Mapping project author,CRISP
Mapping project version,merged
Mapping project version date,2021-04-23
Classification 1 Name,Italian ESCO
Classification 1 Concept type,occupation
Classification 1 Version,1.0.8
Classification 1 Language,it
Classification 1 Landing page,https://ec.europa.eu/esco/portal/occupation
Classification 2 Name,classificazione delle professioni CP2011
Classification 2 Concept type,occupation
Classification 2 Version,2011
Classification 2 Language,it
Classification 2 Landing page,http://professioni.istat.it/cp2011/

In [71]:
cw_it_esco = crosswalks.esco_it_cp2011

#### Checks

- was used for mapping (2980 unique occupations)
- CP-2011 has 829 unique occupations

- a small number of occupations are not labelled in both classifications

In [72]:
for col in cw_it_esco.columns:
    print(col, ":", cw_it_esco.loc[:, col].unique().shape[0])

Classification_1_URI : 2980
Classification_1_PrefLabel : 2978
Classification_1_URL : 2980
Classification_2_ID : 829
Classification_2_PrefLabel : 826
Classification_2_URL : 829
Mapping_relation : 4
Editorial_note : 1


#### Crosswalk ESCO - DE (KldB-2010)

In [73]:
cw_de_esco = crosswalks.esco_de_kldb2010
cw_de_esco

,Classification_1_URI,Classification_1_PrefLabel,Classification_1_URL,Classification_2_ID,Classification_2_PrefLabel,Classification_2_URL,Mapping_relation
0,NaN,NaN,NaN,93551.0,Freiwil. Wehrdienstleistende/r - Laufbahngrupp...,https://berufenet.arbeitsagentur.de/berufenet/...,no relation
1,http://data.europa.eu/esco/occupation/7696b15a...,Artillerieoffizier/Artillerieoffizierin,http://data.europa.eu/esco/occupation/7696b15a...,15341.0,Offizier - Truppendienst,https://berufenet.arbeitsagentur.de/berufenet/...,skos:broadMatch
2,http://data.europa.eu/esco/occupation/262f21a3...,Marineoffizier/Marineoffizierin,http://data.europa.eu/esco/occupation/262f21a3...,15341.0,Offizier - Truppendienst,https://berufenet.arbeitsagentur.de/berufenet/...,skos:broadMatch
3,http://data.europa.eu/esco/occupation/c6a26e11...,Offizier für die Streitkräfte/Offizierin für d...,http://data.europa.eu/esco/occupation/c6a26e11...,15341.0,Offizier - Truppendienst,https://berufenet.arbeitsagentur.de/berufenet/...,skos:exactMatch
4,http://data.europa.eu/esco/occupation/e44b2459...,Kommandeur eines Geschwaders oder einer Kompan...,http://data.europa.eu/esco/occupation/e44b2459...,15341.0,Offizier - Truppendienst,https://berufenet.arbeitsagentur.de/berufenet/...,skos:broadMatch
...,...,...,...,...,...,...,...
4203,http://data.europa.eu/esco/occupation/4d27152a...,Kindergartenhelfer/Kindergartenhelferin,http://data.europa.eu/esco/occupation/4d27152a...,13941.0,Sozialassistent/in,https://berufenet.arbeitsagentur.de/berufenet/...,skos:closeMatch
4204,http://data.europa.eu/esco/occupation/ceda6443...,Sophrologe/Sophrologin,http://data.europa.eu/esco/occupation/ceda6443...,9582.0,Yogalehrer/in,https://berufenet.arbeitsagentur.de/berufenet/...,skos:closeMatch
4205,http://data.europa.eu/esco/occupation/92df4ee3...,Telefonberater Krisen und Notlagen/Telefonbera...,http://data.europa.eu/esco/occupation/92df4ee3...,58699.0,Theologe/Theologin - evangelisch,https://berufenet.arbeitsagentur.de/berufenet/...,skos:broadMatch
4206,http://data.europa.eu/esco/occupation/92df4ee3...,Telefonberater Krisen und Notlagen/Telefonbera...,http://data.europa.eu/esco/occupation/92df4ee3...,58749.0,Theologe/Theologin - katholisch,https://berufenet.arbeitsagentur.de/berufenet/...,skos:broadMatch


In [74]:
# check mapping relation types
cw_de_esco.loc[:, "Mapping_relation"].unique().tolist()

['no relation',
 'skos:broadMatch',
 'skos:exactMatch',
 'skos:narrowMatch',
 'skos:closeMatch']

#### Checks

- ESCO V.1.0.3 was used for mapping (2942 unique occupations)
- KldB-2010 has 2081 unique occupations

- one occupations in ESCO is not labelled

In [75]:
for col in cw_de_esco.columns:
    print(col, ":", cw_de_esco.loc[:, col].unique().shape[0])

Classification_1_URI : 2942
Classification_1_PrefLabel : 2941
Classification_1_URL : 2942
Classification_2_ID : 2081
Classification_2_PrefLabel : 2081
Classification_2_URL : 2081
Mapping_relation : 5


#### Join German mapping on Italian mapping

In [76]:
cw_it_de_reduced = crosswalks.it_cp2011_de_kldb2010()

T:\Documents\Projects\04_jrc_green-skills-regional\03_data-analysis\re4gt\src\data\framework.py:238: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cw_de_esco.loc[:, "Classification_2_ID"] = cw_de_esco.loc[
T:\Documents\Projects\04_jrc_green-skills-regional\03_data-analysis\re4gt\src\data\framework.py:254: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cw_it_de_reduced.loc[:, "Classification_2_ID_de"] = cw_it_de_reduced.loc[


#### Read, inspect and join 5-digit Greenness scores from JRC/Consoli

Note: the crosswalk only includes the following n of occupations:
O*NET-SOC Title, isco08
138, 40


In [77]:
greenness_jrc_raw = onet.green_tasks_narrow_jrc
greenness_jrc_raw

onet.green_occupations_narrow_jrc

,isco08_jrc,occ_eng_jrc,n_green_tasks_jrc,n_tasks_jrc,greenness_jrc
0,1.1.2.4.3,General managers and equivalent in health care,1,11.0,0.090909
1,1.1.2.6.3,Managers and equivalent in health care,1,13.0,0.076923
2,1.2.2.3.0,Directors and general managers of construction...,1,13.0,0.076923
3,1.2.3.2.0,"Directors and managers of the organisation, hu...",1,15.0,0.066667
4,1.2.3.7.0,Directors and managers of the research and dev...,1,12.0,0.083333
5,1.2.3.9.0,Other departmental directors and managers,1,10.0,0.100000
6,1.3.1.2.0,Entrepreneurs and managers of small companies ...,1,14.0,0.071429
7,1.3.1.3.0,Entrepreneurs and managers of small constructi...,1,14.0,0.071429
8,1.3.1.8.0,Entrepreneurs and managers of small companies ...,1,13.0,0.076923
9,2.1.1.2.1,Chemists and related professions,1,14.0,0.071429


Only 117 occupations are covered at the 5-digit level. In O*Net, green task data was
available for 138 occupations. Shouldn't that translate into many more 5-digit
occupations than 117?

In [78]:
print("n occupations unique: ", greenness_jrc_raw.isco08_jrc.unique().shape)
print("n occupations unique: ", greenness_jrc_raw.occ_eng_jrc.unique().shape)
greenness_jrc_raw

n occupations unique:  (117,)
n occupations unique:  (117,)


,isco08_jrc,occ_eng_jrc,task_eng_jrc,n_green_tasks_jrc,n_tasks_jrc,greenness_jrc,comment_eth
0,2.3.1.3.0,Agronomists and foresters,providing advice in the field of animal and pl...,0,11,0.090909,NaN
1,2.3.1.3.0,Agronomists and foresters,designing forestry interventions (reforestatio...,0,11,0.090909,NaN
2,2.3.1.3.0,Agronomists and foresters,prepare land classification and reclamation plans,0,11,0.090909,NaN
3,2.3.1.3.0,Agronomists and foresters,assessing the risks and environmental impact o...,1,11,0.090909,NaN
4,2.3.1.3.0,Agronomists and foresters,managing protected areas or reserves,0,11,0.090909,NaN
...,...,...,...,...,...,...,...
1400,7.2.8.1.0,Workers on packaging machines and packaging of...,load and unload products to be packaged,0,10,0.100000,NaN
1401,7.2.8.1.0,Workers on packaging machines and packaging of...,check labelling and printing of expiry dates,0,10,0.100000,NaN
1402,7.2.8.1.0,Workers on packaging machines and packaging of...,packaging or wrapping materials or products,0,10,0.100000,NaN
1403,7.2.8.1.0,Workers on packaging machines and packaging of...,finishing packages,0,10,0.100000,NaN


Harmonized occ name of "Machinery operators for dry cleaning, bleaching and dyeing of
 industrial yarns and fabrics" (7.2.6.4.0) in excel file


In [79]:
greenness_jrc_raw.columns

Index(['isco08_jrc', 'occ_eng_jrc', 'task_eng_jrc', 'n_green_tasks_jrc',
       'n_tasks_jrc', 'greenness_jrc', 'comment_eth'],
      dtype='object')

In [80]:
# collapse task-level into occupation-level data
greenness_jrc = onet.green_occupations_narrow_jrc

#### Attach 5-digit greenness shares to CW
1. all 117 5-digit occs can be matched to cw
2.

In [81]:
print(greenness_jrc.iloc[0, 0])
print(cw_it_esco.iloc[0, 3])

1.1.2.4.3
1.1.1.1.0


In [82]:
cw_it_de_reduced

,Classification_1_URI,Classification_1_PrefLabel_it,Classification_1_URL_it,Classification_2_ID_it,Classification_2_PrefLabel_it,Classification_2_URL_it,Mapping_relation_it,Editorial_note,Classification_1_PrefLabel_de,Classification_1_URL_de,Classification_2_ID_de,Classification_2_PrefLabel_de,Classification_2_URL_de,Mapping_relation_de
1,http://data.europa.eu/esco/occupation/5bca96fd...,deputato/deputata,http://data.europa.eu/esco/occupation/5bca96fd...,1.1.1.1.0,Membri di organismi di governo e di assemblee ...,http://professioni.istat.it/cp2011/scheda.php?...,skos:broadMatch,NaN,Abgeordneter/Abgeordnete,http://data.europa.eu/esco/occupation/5bca96fd...,90855,Abgeordnete/r,https://berufenet.arbeitsagentur.de/berufenet/...,skos:exactMatch
5,http://data.europa.eu/esco/occupation/7226c10f...,assessore comunale/assessora comunale,http://data.europa.eu/esco/occupation/7226c10f...,1.1.1.3.0,Membri di organismi di governo e di assemblee ...,http://professioni.istat.it/cp2011/scheda.php?...,skos:broadMatch,NaN,Stadtverordneter/Stadtverordnete,http://data.europa.eu/esco/occupation/7226c10f...,90680,Stadtrat/-rätin,https://berufenet.arbeitsagentur.de/berufenet/...,skos:exactMatch
6,http://data.europa.eu/esco/occupation/7226c10f...,assessore comunale/assessora comunale,http://data.europa.eu/esco/occupation/7226c10f...,1.1.1.4.0,Membri di organismi di governo e di assemblee ...,http://professioni.istat.it/cp2011/scheda.php?...,skos:broadMatch,NaN,Stadtverordneter/Stadtverordnete,http://data.europa.eu/esco/occupation/7226c10f...,90680,Stadtrat/-rätin,https://berufenet.arbeitsagentur.de/berufenet/...,skos:exactMatch
7,http://data.europa.eu/esco/occupation/6cee3765...,sindaco/sindaca,http://data.europa.eu/esco/occupation/6cee3765...,1.1.1.4.0,Membri di organismi di governo e di assemblee ...,http://professioni.istat.it/cp2011/scheda.php?...,skos:broadMatch,NaN,Bürgermeister/Bürgermeisterin,http://data.europa.eu/esco/occupation/6cee3765...,90633,Bürgermeister/in,https://berufenet.arbeitsagentur.de/berufenet/...,skos:exactMatch
8,http://data.europa.eu/esco/occupation/a8d05b64...,console,http://data.europa.eu/esco/occupation/a8d05b64...,1.1.2.1.0,"Ambasciatori, ministri plenipotenziari ed alti...",http://professioni.istat.it/cp2011/scheda.php?...,skos:broadMatch,NaN,Konsul/Konsulin,http://data.europa.eu/esco/occupation/a8d05b64...,7609,Beamt(er/in) - Auswärtiger Dienst (höh.Dienst),https://berufenet.arbeitsagentur.de/berufenet/...,skos:broadMatch
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4803,http://data.europa.eu/esco/occupation/7696b15a...,ufficiale di artiglieria,http://data.europa.eu/esco/occupation/7696b15a...,9.1.1.1.0,Ufficiali delle forze armate,http://professioni.istat.it/cp2011/scheda.php?...,skos:broadMatch,NaN,Artillerieoffizier/Artillerieoffizierin,http://data.europa.eu/esco/occupation/7696b15a...,15341,Offizier - Truppendienst,https://berufenet.arbeitsagentur.de/berufenet/...,skos:broadMatch
4810,http://data.europa.eu/esco/occupation/4a3f40a8...,caporal maggiore,http://data.europa.eu/esco/occupation/4a3f40a8...,9.3.1.1.0,Truppa delle forze armate,http://professioni.istat.it/cp2011/scheda.php?...,skos:broadMatch,NaN,Oberstabsgefreiter,http://data.europa.eu/esco/occupation/4a3f40a8...,15474,Soldat/in auf Zeit - Laufbahngruppe Mannschaften,https://berufenet.arbeitsagentur.de/berufenet/...,skos:broadMatch
4812,http://data.europa.eu/esco/occupation/c94dc5b2...,pilota dell’aeronautica militare,http://data.europa.eu/esco/occupation/c94dc5b2...,9.3.1.1.0,Truppa delle forze armate,http://professioni.istat.it/cp2011/scheda.php?...,skos:broadMatch,NaN,Kampfpilot/Kampfpilotin,http://data.europa.eu/esco/occupation/c94dc5b2...,15341,Offizier - Truppendienst,https://berufenet.arbeitsagentur.de/berufenet/...,skos:broadMatch
4814,http://data.europa.eu/esco/occupation/bd5a2f44...,soldato di fanteria,http://data.europa.eu/esco/occupation/bd5a2f44...,9.3.1.1.0,Truppa delle forze armate,http://professioni.istat.it/cp2011/scheda.php

In [83]:
cw_it_de_reduced_gmerged = pd.merge(
    left=cw_it_de_reduced,
    right=greenness_jrc,
    left_on="Classification_2_ID_it",
    right_on="isco08_jrc",
    how="left",
    validate="m:1"
)

# fill nans in greenness col with zeros
cw_it_de_reduced_gmerged["greenness_jrc"] = \
    cw_it_de_reduced_gmerged["greenness_jrc"].fillna(0)

cw_it_de_reduced_gmerged

,Classification_1_URI,Classification_1_PrefLabel_it,Classification_1_URL_it,Classification_2_ID_it,Classification_2_PrefLabel_it,Classification_2_URL_it,Mapping_relation_it,Editorial_note,Classification_1_PrefLabel_de,Classification_1_URL_de,Classification_2_ID_de,Classification_2_PrefLabel_de,Classification_2_URL_de,Mapping_relation_de,isco08_jrc,occ_eng_jrc,n_green_tasks_jrc,n_tasks_jrc,greenness_jrc
0,http://data.europa.eu/esco/occupation/5bca96fd...,deputato/deputata,http://data.europa.eu/esco/occupation/5bca96fd...,1.1.1.1.0,Membri di organismi di governo e di assemblee ...,http://professioni.istat.it/cp2011/scheda.php?...,skos:broadMatch,NaN,Abgeordneter/Abgeordnete,http://data.europa.eu/esco/occupation/5bca96fd...,90855,Abgeordnete/r,https://berufenet.arbeitsagentur.de/berufenet/...,skos:exactMatch,NaN,NaN,NaN,NaN,0.0
1,http://data.europa.eu/esco/occupation/7226c10f...,assessore comunale/assessora comunale,http://data.europa.eu/esco/occupation/7226c10f...,1.1.1.3.0,Membri di organismi di governo e di assemblee ...,http://professioni.istat.it/cp2011/scheda.php?...,skos:broadMatch,NaN,Stadtverordneter/Stadtverordnete,http://data.europa.eu/esco/occupation/7226c10f...,90680,Stadtrat/-rätin,https://berufenet.arbeitsagentur.de/berufenet/...,skos:exactMatch,NaN,NaN,NaN,NaN,0.0
2,http://data.europa.eu/esco/occupation/7226c10f...,assessore comunale/assessora comunale,http://data.europa.eu/esco/occupation/7226c10f...,1.1.1.4.0,Membri di organismi di governo e di assemblee ...,http://professioni.istat.it/cp2011/scheda.php?...,skos:broadMatch,NaN,Stadtverordneter/Stadtverordnete,http://data.europa.eu/esco/occupation/7226c10f...,90680,Stadtrat/-rätin,https://berufenet.arbeitsagentur.de/berufenet/...,skos:exactMatch,NaN,NaN,NaN,NaN,0.0
3,http://data.europa.eu/esco/occupation/6cee3765...,sindaco/sindaca,http://data.europa.eu/esco/occupation/6cee3765...,1.1.1.4.0,Membri di organismi di governo e di assemblee ...,http://professioni.istat.it/cp2011/scheda.php?...,skos:broadMatch,NaN,Bürgermeister/Bürgermeisterin,http://data.europa.eu/esco/occupation/6cee3765...,90633,Bürgermeister/in,https://berufenet.arbeitsagentur.de/berufenet/...,skos:exactMatch,NaN,NaN,NaN,NaN,0.0
4,http://data.europa.eu/esco/occupation/a8d05b64...,console,http://data.europa.eu/esco/occupation/a8d05b64...,1.1.2.1.0,"Ambasciatori, ministri plenipotenziari ed alti...",http://professioni.istat.it/cp2011/scheda.php?...,skos:broadMatch,NaN,Konsul/Konsulin,http://data.europa.eu/esco/occupation/a8d05b64...,7609,Beamt(er/in) - Auswärtiger Dienst (höh.Dienst),https://berufenet.arbeitsagentur.de/berufenet/...,skos:broadMatch,NaN,NaN,NaN,NaN,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4611,http://data.europa.eu/esco/occupation/7696b15a...,ufficiale di artiglieria,http://data.europa.eu/esco/occupation/7696b15a...,9.1.1.1.0,Ufficiali delle forze armate,http://professioni.istat.it/cp2011/scheda.php?...,skos:broadMatch,NaN,Artillerieoffizier/Artillerieoffizierin,http://data.europa.eu/esco/occupation/7696b15a...,15341,Offizier - Truppendienst,https://berufenet.arbeitsagentur.de/berufenet/...,skos:broadMatch,NaN,NaN,NaN,NaN,0.0
4612,http://data.europa.eu/esco/occupation/4a3f40a8...,caporal maggiore,http://data.europa.eu/esco/occupation/4a3f40a8...,9.3.1.1.0,Truppa delle forze armate,http://professioni.istat.it/cp2011/scheda.php?...,skos:broadMatch,NaN,Oberstabsgefreiter,http://data.europa.eu/esco/occupation/4a3f40a8...,15474,Soldat/in auf Zeit - Laufbahngruppe Mannschaften,https://berufenet.arbeitsagentur.de/berufenet/...,skos:broadMatch,NaN,NaN,NaN,NaN,0.0
4613,http://data.europa.eu/esco/occupation/c94dc5b2...,pilota dell’aeronautica militare,http://data.europa.eu/esco/occupation/c94dc5b2...,9.3.1.1.0,Truppa delle forze armate,http://professioni.istat.it/cp2011/scheda.php?...,skos:broadMatch,NaN,Kampfpilot/Kampfpilotin,http://data.europa.eu/esco/occupation/c94dc5b2...,15341,Offizier - Truppendienst,https://berufenet.arbeitsagentur.de/berufenet

all but one 5-digit occ could be mapped to cw

In [84]:
cw_it_de_reduced_gmerged.isco08_jrc.unique().shape

(117,)

Greennest occupations in KldB

In [85]:
cw_it_de_reduced_gmerged[["Classification_1_PrefLabel_de", "greenness_jrc"]].replace\
    (0, np.nan).dropna()

,Classification_1_PrefLabel_de,greenness_jrc
21,Leiter einer Gesundheitseinrichtung/Leiterin e...,0.090909
22,Leiter einer Gesundheitseinrichtung/Leiterin e...,0.090909
23,Leiter eines medizinischen Labors/Leiterin ein...,0.090909
24,Patientendatenverwalter/Patientendatenverwalterin,0.090909
25,Patientendatenverwalter/Patientendatenverwalterin,0.090909
...,...,...
4542,Küster/Küsterin,0.076923
4543,Friedhofsarbeiter/Friedhofsarbeiterin,0.076923
4547,Garderobenaufseher/Garderobenaufseherin,0.083333
4548,Kabinenwart/Kabinenwartin,0.083333


#### Save final ds

In [86]:
cw_it_de_reduced_gmerged.to_csv(
    os.path.join(useful_paths.data_interim, "crosswalks",
                 "crosswalk_it_cp2011_de_kldb2010_reduced_with_metadata.csv")
)